
# Daily Challenge — Churn Prediction (Solution Notebook)


This notebook delivers a clean, production-style pipeline for **customer churn prediction**:
1. Setup & Data Load  
2. Exploratory Data Analysis (EDA)  
3. Preprocessing & Feature Engineering (ColumnTransformer)  
4. Baselines & Models (Dummy, Logistic Regression, Random Forest, Gradient Boosting)  
5. Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Confusion Matrix, ROC & PR curves)  
6. Feature Importance & Coefficients  
7. Threshold Tuning & Business Targeting  
8. Insights & Reflection

> Notes
> - Charts use **matplotlib only**.
> - The pipeline is **train/test split with stratification** and **scikit-learn Pipelines**.
> - If `Churn_Modelling.csv` is missing, the notebook synthesizes a realistic dataset.



## 1) Setup & Data Loading
- Import libraries
- Load `Churn_Modelling.csv` if present, else generate synthetic data
- Inspect schema and basic stats


In [ ]:

# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc, classification_report
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_PATH = '/mnt/data/Churn_Modelling.csv'
TARGET_CANDIDATES = ['Exited', 'Churn', 'HasChurned', 'churn', 'target']

def synthesize_churn(n=10000, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    df = pd.DataFrame({
        'CustomerId': rng.integers(1_000_000, 9_999_999, size=n),
        'Surname': rng.choice(['Smith','Cohen','Levi','Nguyen','Martin','Garcia','Rosen','Katz','Haddad','Dubois'], size=n),
        'CreditScore': rng.normal(650, 100, size=n).clip(300, 850).round().astype(int),
        'Geography': rng.choice(['France','Spain','Germany'], size=n, p=[0.45,0.25,0.30]),
        'Gender': rng.choice(['Male','Female'], size=n),
        'Age': rng.normal(39, 10, size=n).clip(18, 92).round().astype(int),
        'Tenure': rng.integers(0, 11, size=n),
        'Balance': rng.gamma(3.0, 2500, size=n),  # skewed positive
        'NumOfProducts': rng.choice([1,2,3,4], size=n, p=[0.55,0.35,0.07,0.03]),
        'HasCrCard': rng.choice([0,1], size=n, p=[0.3,0.7]),
        'IsActiveMember': rng.choice([0,1], size=n, p=[0.48,0.52]),
        'EstimatedSalary': rng.normal(100000, 40000, size=n).clip(10000,300000),
    })
    # log-transformed proxies
    log_bal = np.log1p(df['Balance'])
    act = df['IsActiveMember']
    score = df['CreditScore'] / 850.0
    age = (df['Age'] - 40)/20.0
    products = df['NumOfProducts']
    germany = (df['Geography'] == 'Germany').astype(int)
    female = (df['Gender'] == 'Female').astype(int)
    # latent propensity
    lin = -1.0 - 0.8*act - 0.7*score + 0.4*age + 0.3*(products==1) + 0.3*germany + 0.1*female - 0.2*log_bal
    p = 1/(1+np.exp(-lin))
    df['Exited'] = (rng.random(n) < p).astype(int)
    return df

# Load or synthesize
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    df = synthesize_churn(10000)

print('Shape:', df.shape)
display(df.head(10))

print('\nColumns:', list(df.columns))
print('\nInfo:')
display(df.info())

# Find target column
target_col = None
for c in TARGET_CANDIDATES:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    # try to infer: common Kaggle file uses 'Exited'
    target_col = 'Exited' if 'Exited' in df.columns else df.columns[-1]
print('Target column assumed as:', target_col)

# Basic describe
display(df.describe(include='all').T.head(30))



## 2) Exploratory Data Analysis (EDA)
- Missing values, class balance
- Simple distributions (via describe) and value_counts for categoricals


In [ ]:

# Missing values
print('Missing values per column:')
print(df.isnull().sum())

# Class balance
if target_col in df.columns:
    print('\nClass balance:')
    print(df[target_col].value_counts(dropna=False))
    pos_rate = df[target_col].mean()
    print(f'Positive class rate (churn=1): {pos_rate:.3f}')

# Quick look at common categorical columns
cat_candidates = ['Geography','Gender']
for col in cat_candidates:
    if col in df.columns:
        print(f'\nValue counts for {col}:')
        print(df[col].value_counts())

# Simple histogram for age / creditscore if present (matplotlib only)
for col in ['Age','CreditScore','Balance','EstimatedSalary']:
    if col in df.columns and np.issubdtype(df[col].dtype, np.number):
        plt.figure(figsize=(6,4))
        plt.hist(df[col].dropna(), bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()



## 3) Preprocessing & Feature Engineering
- Split train/test with **stratify**
- Build a `ColumnTransformer` to:
  - One-hot encode categoricals
  - Scale numeric features
- Assemble into a `Pipeline`


In [ ]:

# Separate features/target
y = df[target_col].astype(int).values
X = df.drop(columns=[target_col])

# Identify feature types
numeric_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.number)]
# Heuristic: non-numeric (object/category/bool treated as categorical)
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# Some ID-like columns can be noisy; drop obvious IDs if present
for id_col in ['RowNumber','CustomerId','Surname','Id']:
    if id_col in X.columns:
        X = X.drop(columns=[id_col])
        if id_col in numeric_cols:
            numeric_cols.remove(id_col)
        if id_col in categorical_cols:
            categorical_cols.remove(id_col)

print('Numeric cols:', numeric_cols[:10], '... total:', len(numeric_cols))
print('Categorical cols:', categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocess
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)



## 4) Baselines & Models
We compare a **Dummy** baseline with:
- **Logistic Regression**
- **Random Forest**
- **Gradient Boosting**

All models are wrapped in the same preprocessing pipeline.


In [ ]:

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    proba = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:,1]
    elif hasattr(model, 'decision_function'):
        # Map decision scores to [0,1] via min-max for PR/ROC curves if needed
        scores = model.decision_function(X_test)
        # Fall back to ranking-based mapping (not calibrated)
        smin, smax = scores.min(), scores.max()
        proba = (scores - smin) / (smax - smin + 1e-12)
    else:
        # As last resort, use predictions as probas 0/1
        proba = model.predict(X_test)

    preds = (proba >= 0.5).astype(int)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    roc = roc_auc_score(y_test, proba)

    print(f'[{name}]  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {roc:.4f}')
    return {
        'name': name, 'model': model, 'proba': proba, 'preds': preds,
        'metrics': {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': roc}
    }

models = []

# Dummy baseline
dummy_clf = Pipeline(steps=[('prep', preprocessor), ('clf', DummyClassifier(strategy='stratified', random_state=42))])
models.append(('Dummy', dummy_clf))

# Logistic Regression
log_reg = Pipeline(steps=[('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, n_jobs=None, random_state=42))])
models.append(('LogisticRegression', log_reg))

# Random Forest
rf_clf = Pipeline(steps=[('prep', preprocessor), ('clf', RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_split=2, min_samples_leaf=1,
    random_state=42, n_jobs=-1
))])
models.append(('RandomForest', rf_clf))

# Gradient Boosting
gb_clf = Pipeline(steps=[('prep', preprocessor), ('clf', GradientBoostingClassifier(random_state=42))])
models.append(('GradientBoosting', gb_clf))

results = []
for name, pipe in models:
    res = evaluate_model(name, pipe, X_train, y_train, X_test, y_test)
    results.append(res)

# Summary table
summary = pd.DataFrame([
    dict(Model=r['name'], **r['metrics']) for r in results
]).sort_values('roc_auc', ascending=False)
display(summary)



## 5) Evaluation — Confusion Matrix, ROC & PR Curves


In [ ]:

# Pick best by ROC-AUC
best = max(results, key=lambda r: r['metrics']['roc_auc'])
best_name = best['name']; best_model = best['model']; best_proba = best['proba']
best_preds = best['preds']

print('Best model:', best_name)
print(classification_report(y_test, best_preds, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(4,4))
plt.imshow(cm, cmap=None)  # default colormap
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.tight_layout()
plt.show()

# ROC Curve (plot for top 3 by ROC-AUC)
top3 = sorted(results, key=lambda r: r['metrics']['roc_auc'], reverse=True)[:3]
plt.figure(figsize=(6,4))
for r in top3:
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    roc_auc = roc_auc_score(y_test, r['proba'])
    plt.plot(fpr, tpr, label=f"{r['name']} (AUC={roc_auc:.3f})")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (Top 3)')
plt.legend()
plt.tight_layout()
plt.show()

# Precision-Recall Curve (Top 3)
plt.figure(figsize=(6,4))
for r in top3:
    precision, recall, _ = precision_recall_curve(y_test, r['proba'])
    pr_auc = auc(recall, precision)
    plt.plot(recall, precision, label=f"{r['name']} (AUC={pr_auc:.3f})")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves (Top 3)')
plt.legend()
plt.tight_layout()
plt.show()



## 6) Feature Importance & Coefficients
- For **Random Forest / Gradient Boosting**: model-based importance  
- For **Logistic Regression**: absolute coefficients (on one-hot/standardized space)
> We recover transformed feature names to attribute importances.


In [ ]:

# Helper to extract feature names after ColumnTransformer
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    num_features = list(numeric_cols)
    cat = preprocessor.named_transformers_['cat']
    if hasattr(cat, 'get_feature_names_out'):
        cat_features = list(cat.get_feature_names_out(categorical_cols))
    else:
        # Fallback: generic names
        cat_features = [f'cat_{i}' for i in range(len(categorical_cols))]
    return num_features + cat_features

# Fit best model on full training for importance extraction
best_model.fit(X_train, y_train)

# Extract feature names
prep = best_model.named_steps['prep']
feat_names = get_feature_names(prep, numeric_cols, categorical_cols)

if 'RandomForest' in best_name or 'GradientBoosting' in best_name:
    clf = best_model.named_steps[[k for k in best_model.named_steps if k != 'prep'][0]]
    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
        imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False).head(25)
        display(imp_df)
        # Bar plot
        plt.figure(figsize=(8,6))
        y_pos = np.arange(len(imp_df))
        plt.barh(y_pos, imp_df['importance'][::-1])
        plt.yticks(y_pos, imp_df['feature'][::-1])
        plt.xlabel('Importance')
        plt.title(f'Feature Importances — {best_name} (Top 25)')
        plt.tight_layout()
        plt.show()
elif 'LogisticRegression' in best_name:
    clf = best_model.named_steps[[k for k in best_model.named_steps if k != 'prep'][0]]
    coefs = np.abs(clf.coef_[0])
    coef_df = pd.DataFrame({'feature': feat_names, 'abs_coef': coefs}).sort_values('abs_coef', ascending=False).head(25)
    display(coef_df)
    plt.figure(figsize=(8,6))
    y_pos = np.arange(len(coef_df))
    plt.barh(y_pos, coef_df['abs_coef'][::-1])
    plt.yticks(y_pos, coef_df['feature'][::-1])
    plt.xlabel('|Coefficient|')
    plt.title(f'Logistic Coefficients — {best_name} (Top 25)')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance not available for this model.')



## 7) Threshold Tuning & Business Targeting
We vary the classification threshold to trade precision vs recall (e.g., marketing budget targeting top-K).


In [ ]:

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    preds_t = (best_proba >= t).astype(int)
    rows.append({
        'threshold': t,
        'precision': precision_score(y_test, preds_t, zero_division=0),
        'recall': recall_score(y_test, preds_t, zero_division=0),
        'f1': f1_score(y_test, preds_t, zero_division=0),
        'positives_rate': preds_t.mean()
    })

thr_df = pd.DataFrame(rows)
display(thr_df)

# Plot F1 vs threshold
plt.figure(figsize=(6,4))
plt.plot(thr_df['threshold'], thr_df['f1'], marker='o')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title(f'F1 vs Threshold — {best_name}')
plt.tight_layout()
plt.show()

# Example: target top 20% by score
k = int(0.2 * len(best_proba))
idx_sorted = np.argsort(-best_proba)
selected = idx_sorted[:k]
lift = y_test[selected].mean() / y_test.mean()
print(f'If you target top 20% by score, positive rate in segment = {y_test[selected].mean():.3f} (lift x{lift:.2f} vs overall {y_test.mean():.3f})')



## 8) Insights & Reflection

**Insights examples:**
- **Model choice:** We selected the best model by ROC-AUC and cross-checked F1.  
- **Business lift:** Targeting top X% by predicted risk significantly increases conversion (churn capture).  
- **Drivers:** Feature importance suggests which factors correlate with churn (e.g., inactivity, geography, few products).  
- **Next steps:** Calibrate probabilities, add recent transaction features, costs, and run A/B tests for retention offers.

**Reflection questions (templates):**
1. *Which preprocessing steps mattered most and why?*  
2. *Why did the best model outperform others on ROC-AUC?*  
3. *How would you set the threshold under different cost assumptions?*  
4. *What data would you add to improve the model?*  
5. *How would you monitor data drift and performance over time?*
